# Create Eval1 subset

This notebook creates a reproducible subset of 30 queries from the Biomni Eval1 dataset.

The goal is to:
- fix the evaluation dataset
- ensure reproducibility across agent comparisons
- store the subset as a CSV file

Output:
'data_subset/project_subset.csv'

In [1]:
import random
import csv
from pathlib import Path
import pandas as pd
from datasets import load_dataset

#### Auxiliary functions

In [2]:
def clean_text(text):
    if not isinstance(text, str):
        return text

    return (
        text.replace("\n", " ")
            .replace("\r", " ")
            .replace("\t", " ")
            .replace("  ", " ")
            .strip()
    )

#### Load Dataset

In [3]:
dataset = load_dataset("biomni/Eval1", split="test")
rows = list(dataset)

print("Total rows:", len(rows))
print("Columns:", dataset.column_names)

Total rows: 433
Columns: ['instance_id', 'task_instance_id', 'prompt', 'task_name', 'split', 'answer']


#### Configuration

In [4]:
sample_size = 30
seed = 42

output_path = Path("project_subset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

#### Sample queries

In [5]:
rng = random.Random(seed)
sampled_rows = rng.sample(rows, sample_size)

subset = []

for i, row in enumerate(sampled_rows, start=1):
    subset.append(
        {
            "sample_index": i,
            "task_name": row.get("task_name", ""),
            "task_instance_id": row.get("task_instance_id", row.get("instance_id", "")),
            "input_query": clean_text(row.get("prompt", "")),
            "dataset_eval1_answer": clean_text(row.get("answer", "")),
        }
    )

df = pd.DataFrame(subset)

df.head()

,sample_index,task_name,task_instance_id,input_query,dataset_eval1_answer
0,1,lab_bench_dbqa,387,The following is a multiple choice question ab...,A
1,2,screen_gene_retrieval,259,Your task is to identify the gene with the str...,DHODH
2,3,gwas_variant_prioritization,88,Your task is to identify the most promising va...,rs2160860
3,4,gwas_causal_gene_opentargets,27,Your task is to identify likely causal genes w...,TBX4
4,5,lab_bench_seqqa,187,The following is a multiple choice question ab...,D


#### Save subset

In [6]:
df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

print("Saved to:", output_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Saved to: project_subset.csv
Rows: 30
Columns: ['sample_index', 'task_name', 'task_instance_id', 'input_query', 'dataset_eval1_answer']
